# Desktop Helper — Fine-tuning on Colab

Fine-tunes `DesktopHelperLM` (OPT-350M weights transplanted into the custom architecture) on Dolly + synthetic tool-call data.

**Prerequisites in Google Drive:**
- `opt_transplant.pt` — the transplanted checkpoint (produced locally by `model/load_opt.py`)

**Runtime:** set to GPU (Runtime → Change runtime type → T4 GPU or better).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo (pretrained-model branch)

In [ ]:
!git clone https://github.com/JaysonSalemmo/Desktop_Helper.git
%cd Desktop_Helper
!git checkout pretrained-model

## 3. Install dependencies

Colab ships with torch + CUDA. We only need the data/tokenizer libraries.

In [ ]:
!pip install -q datasets tokenizers tensorboard

## 4. Regenerate the tool-call data

`data/tool_calls.jsonl` is gitignored, so regenerate it here (deterministic — same seed as local).

In [ ]:
!python -m model.data.tool_calls --count 8000 --seed 42 --output data/tool_calls.jsonl

## 5. Confirm GPU is available

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → GPU'
print(torch.cuda.get_device_name(0))

## 6. Launch TensorBoard (run this *before* training)

Starts a live dashboard that auto-refreshes as training writes loss/LR curves to Drive. Run this cell, then run the training cell below — the charts update in real time. If it shows "No dashboards" at first, that's expected until the training cell starts writing.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/desktop_helper_checkpoints/runs

## 7. Fine-tune

Adjust `--checkpoint` to wherever you uploaded `opt_transplant.pt` in Drive.
Checkpoints save to Drive after every epoch, so they survive session resets. TensorBoard logs write to `<output>/runs` — the same path the cell above watches.

If you hit CUDA out-of-memory on a T4, lower `--batch-size` to 2 and raise `--grad-accum` to 16 (keeps the effective batch at 32).

In [ ]:
!python -m model.train \
  --checkpoint /content/drive/MyDrive/opt_transplant.pt \
  --tokenizer model/tokenizer.json \
  --tool-calls data/tool_calls.jsonl \
  --output /content/drive/MyDrive/desktop_helper_checkpoints/ \
  --epochs 16 \
  --batch-size 4 \
  --grad-accum 8

## 8. Faithfulness eval

Scores how well the checkpoint copies facts from injected `[RESULT]` blocks (held-out entities — see `model/eval_faithfulness.py`). Baseline epoch_08 from the previous run scored near zero; this retrain (RESULT-masked loss + high-entropy result content) is aimed squarely at this number. Run it on the final epoch, and compare a couple of earlier epochs if curious.

In [ ]:
!python -m model.eval_faithfulness --checkpoint /content/drive/MyDrive/desktop_helper_checkpoints/epoch_16.pt